# Analyze Display Name Alternatives Pollution in Authors

When author disambiguation mis-attributes a work, the wrong person's name gets mixed into
`display_name_alternatives`. This corrupts `longest_name` → `parsed_longest_name` → `block_key`,
potentially cascading further mis-attribution.

This notebook measures the scope of that pollution using multiple complementary signals.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

# Auto-detect environment: Databricks (spark.sql) vs local (utils.databricks_sql)
try:
    spark  # noqa: F821
    _ON_DATABRICKS = True
except NameError:
    _ON_DATABRICKS = False

if _ON_DATABRICKS:
    print("Running on Databricks — using spark.sql()")
    def run_query_df(sql, size=None):
        return spark.sql(sql).toPandas()  # noqa: F821
else:
    import sys
    sys.path.insert(0, '../..')
    from utils.databricks_sql import run_query_df as _local_run_query_df
    print("Running locally — using utils.databricks_sql")
    def run_query_df(sql, size="xlarge"):
        return _local_run_query_df(sql, size=size)

## Pre-Step: Check for ORCID Canonical Names Table

In [ ]:
# Search for orcid-related tables across all schemas
orcid_tables = run_query_df("""
    SELECT table_schema, table_name
    FROM openalex.information_schema.tables
    WHERE LOWER(table_name) LIKE '%orcid%'
    ORDER BY table_schema, table_name
""")
print("ORCID-related tables:")
print(orcid_tables.to_string())

In [ ]:
# Only table found: openalex.mid.author_orcid (author_id, orcid, updated, evidence)
# No ORCID canonical display names available — can't use as ground truth for name validation.
# We'll rely on parsed_names_lookup comparisons instead.
print("No ORCID canonical name table found. Proceeding with parsed-name analysis.")

## Step 1: Baseline Counts

In [ ]:
baseline = run_query_df("""
    SELECT
        COUNT(*) AS total_authors,
        COUNT_IF(display_name_alternatives IS NOT NULL AND SIZE(display_name_alternatives) > 0) AS authors_with_alternatives,
        COUNT_IF(orcid IS NOT NULL) AS authors_with_orcid,
        COUNT_IF(orcid IS NOT NULL AND display_name_alternatives IS NOT NULL AND SIZE(display_name_alternatives) > 0) AS orcid_with_alternatives
    FROM openalex.authors.openalex_authors
""", size="xlarge")
print("Baseline counts:")
print(baseline.to_string(index=False))

In [ ]:
# Distribution of alternative counts per author
alt_dist = run_query_df("""
    SELECT
        COALESCE(SIZE(display_name_alternatives), 0) AS num_alternatives,
        COUNT(*) AS author_count
    FROM openalex.authors.openalex_authors
    GROUP BY 1
    ORDER BY 1
""", size="xlarge")
print("Distribution of display_name_alternatives count:")
print(alt_dist.to_string(index=False))

## Step 2: Verify Known Example (Author 5059085934)

In [ ]:
# Deep dive on the known polluted author
example = run_query_df("""
    SELECT
        id,
        display_name,
        orcid,
        display_name_alternatives,
        longest_name,
        parsed_longest_name,
        block_key
    FROM openalex.authors.openalex_authors
    WHERE id = 5059085934
""", size="xlarge")
print("Author 5059085934:")
for col in example.columns:
    print(f"  {col}: {example[col].iloc[0]}")

In [ ]:
# Trace which works contributed each alternative name
example_works = run_query_df("""
    SELECT
        author_display_name,
        raw_author_name,
        work_id,
        pub_year
    FROM openalex.authors.work_authorships_staging
    WHERE author_id = 'https://openalex.org/A5059085934'
    ORDER BY pub_year DESC
""", size="xlarge")
print(f"Works attributed to author 5059085934: {len(example_works)}")
print("\nDistinct author_display_name values:")
print(example_works['author_display_name'].value_counts().to_string())
print("\nDistinct raw_author_name values:")
print(example_works['raw_author_name'].value_counts().to_string())

## Step 3: Within-Author Last-Name Divergence (Primary Signal)

For each author, explode alternatives, join to `parsed_names_lookup` to get each
alternative's parsed last name, and count distinct last names that differ from
the author's own `parsed_longest_name.last`.

In [ ]:
# Distribution of foreign last names per author
foreign_ln_dist = run_query_df("""
    WITH author_alts AS (
        SELECT
            a.id,
            a.parsed_longest_name.last AS author_last,
            alt.col AS alt_name
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW OUTER EXPLODE(a.display_name_alternatives) alt
        WHERE a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
    ),
    alt_parsed AS (
        SELECT
            aa.id,
            aa.author_last,
            pn.parsed_name.last AS alt_last
        FROM author_alts aa
        LEFT JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(aa.alt_name) = pn.raw_author_name
    ),
    foreign_counts AS (
        SELECT
            id,
            COUNT(DISTINCT CASE
                WHEN alt_last IS NOT NULL
                 AND author_last IS NOT NULL
                 AND alt_last != author_last
                THEN alt_last
            END) AS num_foreign_last_names
        FROM alt_parsed
        GROUP BY id
    )
    SELECT
        num_foreign_last_names,
        COUNT(*) AS author_count
    FROM foreign_counts
    GROUP BY 1
    ORDER BY 1
""", size="xlarge")
print("Distribution of num_foreign_last_names:")
print(foreign_ln_dist.to_string(index=False))

In [ ]:
# Sample: ORCID authors with 2+ foreign last names for manual verification
foreign_ln_sample = run_query_df("""
    WITH author_alts AS (
        SELECT
            a.id,
            a.display_name,
            a.orcid,
            a.parsed_longest_name.last AS author_last,
            a.display_name_alternatives,
            alt.col AS alt_name
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW OUTER EXPLODE(a.display_name_alternatives) alt
        WHERE a.orcid IS NOT NULL
          AND a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
    ),
    alt_parsed AS (
        SELECT
            aa.id,
            aa.display_name,
            aa.orcid,
            aa.author_last,
            aa.display_name_alternatives,
            aa.alt_name,
            pn.parsed_name.last AS alt_last
        FROM author_alts aa
        LEFT JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(aa.alt_name) = pn.raw_author_name
    ),
    foreign_counts AS (
        SELECT
            id,
            ANY_VALUE(display_name) AS display_name,
            ANY_VALUE(orcid) AS orcid,
            ANY_VALUE(author_last) AS author_last,
            ANY_VALUE(display_name_alternatives) AS display_name_alternatives,
            COLLECT_SET(CASE
                WHEN alt_last IS NOT NULL
                 AND author_last IS NOT NULL
                 AND alt_last != author_last
                THEN alt_last
            END) AS foreign_last_names,
            COUNT(DISTINCT CASE
                WHEN alt_last IS NOT NULL
                 AND author_last IS NOT NULL
                 AND alt_last != author_last
                THEN alt_last
            END) AS num_foreign_last_names
        FROM alt_parsed
        GROUP BY id
    )
    SELECT
        id, display_name, orcid, author_last,
        display_name_alternatives, foreign_last_names, num_foreign_last_names
    FROM foreign_counts
    WHERE num_foreign_last_names >= 2
    ORDER BY num_foreign_last_names DESC
    LIMIT 25
""", size="xlarge")
print(f"Sample ORCID authors with 2+ foreign last names ({len(foreign_ln_sample)} rows):")
print(foreign_ln_sample.to_string(index=False))

## Step 4: Block Key Corruption Detection

Compare `parsed_longest_name.last` against the parsed last name of `display_name`.
When they differ, the block_key was derived from a polluted name.

In [ ]:
# Count block_key mismatches
block_key_stats = run_query_df("""
    SELECT
        COUNT(*) AS total_with_both_parsed,
        COUNT_IF(a.parsed_longest_name.last != pn_dn.parsed_name.last) AS block_key_mismatches,
        COUNT_IF(a.parsed_longest_name.last != pn_dn.parsed_name.last AND a.orcid IS NOT NULL) AS mismatches_with_orcid,
        ROUND(
            100.0 * COUNT_IF(a.parsed_longest_name.last != pn_dn.parsed_name.last) / COUNT(*),
            2
        ) AS mismatch_pct
    FROM openalex.authors.openalex_authors a
    INNER JOIN openalex.authors.parsed_names_lookup pn_dn
        ON TRIM(a.display_name) = pn_dn.raw_author_name
    WHERE a.parsed_longest_name.last IS NOT NULL
      AND pn_dn.parsed_name.last IS NOT NULL
""", size="xlarge")
print("Block key corruption:")
print(block_key_stats.to_string(index=False))

In [ ]:
# Sample ORCID authors with mismatched block_keys
block_key_sample = run_query_df("""
    SELECT
        a.id,
        a.display_name,
        a.orcid,
        a.longest_name,
        a.block_key,
        a.parsed_longest_name.last AS longest_name_last,
        pn_dn.parsed_name.last AS display_name_last,
        a.display_name_alternatives
    FROM openalex.authors.openalex_authors a
    INNER JOIN openalex.authors.parsed_names_lookup pn_dn
        ON TRIM(a.display_name) = pn_dn.raw_author_name
    WHERE a.parsed_longest_name.last IS NOT NULL
      AND pn_dn.parsed_name.last IS NOT NULL
      AND a.parsed_longest_name.last != pn_dn.parsed_name.last
      AND a.orcid IS NOT NULL
    LIMIT 25
""", size="xlarge")
print(f"Sample ORCID authors with mismatched block_keys ({len(block_key_sample)} rows):")
print(block_key_sample.to_string(index=False))

## Step 5: ORCID-Subset Validation

Repeat foreign last name analysis restricted to ORCID authors for a cleaner pollution rate.

In [ ]:
orcid_foreign_ln = run_query_df("""
    WITH author_alts AS (
        SELECT
            a.id,
            a.parsed_longest_name.last AS author_last,
            alt.col AS alt_name
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW OUTER EXPLODE(a.display_name_alternatives) alt
        WHERE a.orcid IS NOT NULL
          AND a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
    ),
    alt_parsed AS (
        SELECT
            aa.id,
            aa.author_last,
            pn.parsed_name.last AS alt_last
        FROM author_alts aa
        LEFT JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(aa.alt_name) = pn.raw_author_name
    ),
    foreign_counts AS (
        SELECT
            id,
            COUNT(DISTINCT CASE
                WHEN alt_last IS NOT NULL
                 AND author_last IS NOT NULL
                 AND alt_last != author_last
                THEN alt_last
            END) AS num_foreign_last_names
        FROM alt_parsed
        GROUP BY id
    )
    SELECT
        num_foreign_last_names,
        COUNT(*) AS author_count
    FROM foreign_counts
    GROUP BY 1
    ORDER BY 1
""", size="xlarge")
print("ORCID authors — distribution of num_foreign_last_names:")
print(orcid_foreign_ln.to_string(index=False))

total_orcid = orcid_foreign_ln['author_count'].sum()
polluted_orcid = orcid_foreign_ln[orcid_foreign_ln['num_foreign_last_names'] >= 1]['author_count'].sum()
print(f"\nORCID pollution rate: {polluted_orcid}/{total_orcid} = {100*polluted_orcid/total_orcid:.2f}%")

## Step 6: Levenshtein Distance Distribution

Compute normalized Levenshtein distance between `display_name` and each alternative.
Alternatives with >60% normalized distance are suspicious.

In [ ]:
lev_dist = run_query_df("""
    WITH alt_distances AS (
        SELECT
            a.id,
            a.display_name,
            alt.col AS alt_name,
            levenshtein(LOWER(a.display_name), LOWER(alt.col)) AS lev_dist,
            GREATEST(LENGTH(a.display_name), LENGTH(alt.col)) AS max_len,
            CASE
                WHEN GREATEST(LENGTH(a.display_name), LENGTH(alt.col)) = 0 THEN 0.0
                ELSE ROUND(
                    levenshtein(LOWER(a.display_name), LOWER(alt.col))
                    / GREATEST(LENGTH(a.display_name), LENGTH(alt.col)),
                    2
                )
            END AS norm_dist
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW EXPLODE(a.display_name_alternatives) alt
        WHERE a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
    )
    SELECT
        CASE
            WHEN norm_dist <= 0.1 THEN '0.0-0.1'
            WHEN norm_dist <= 0.2 THEN '0.1-0.2'
            WHEN norm_dist <= 0.3 THEN '0.2-0.3'
            WHEN norm_dist <= 0.4 THEN '0.3-0.4'
            WHEN norm_dist <= 0.5 THEN '0.4-0.5'
            WHEN norm_dist <= 0.6 THEN '0.5-0.6'
            WHEN norm_dist <= 0.7 THEN '0.6-0.7'
            WHEN norm_dist <= 0.8 THEN '0.7-0.8'
            WHEN norm_dist <= 0.9 THEN '0.8-0.9'
            ELSE '0.9-1.0'
        END AS distance_bucket,
        COUNT(*) AS pair_count,
        COUNT(DISTINCT id) AS author_count
    FROM alt_distances
    GROUP BY 1
    ORDER BY 1
""", size="xlarge")
print("Normalized Levenshtein distance distribution (display_name vs each alternative):")
print(lev_dist.to_string(index=False))

total_pairs = lev_dist['pair_count'].sum()
suspicious = lev_dist[lev_dist['distance_bucket'] >= '0.6']['pair_count'].sum()
print(f"\nSuspicious (>0.6 norm distance): {suspicious}/{total_pairs} = {100*suspicious/total_pairs:.2f}%")

## Step 7: Cross-Author Name Matching

For alternatives flagged as foreign (different parsed last name), check if they exactly
match the `display_name` of a different ORCID author. This directly shows work mis-attribution.

In [ ]:
cross_match = run_query_df("""
    WITH foreign_alts AS (
        SELECT
            a.id AS source_author_id,
            a.display_name AS source_display_name,
            a.orcid AS source_orcid,
            a.parsed_longest_name.last AS source_last,
            alt.col AS alt_name,
            pn.parsed_name.last AS alt_last
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW EXPLODE(a.display_name_alternatives) alt
        INNER JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(alt.col) = pn.raw_author_name
        WHERE a.orcid IS NOT NULL
          AND a.parsed_longest_name.last IS NOT NULL
          AND pn.parsed_name.last IS NOT NULL
          AND a.parsed_longest_name.last != pn.parsed_name.last
    )
    SELECT
        fa.source_author_id,
        fa.source_display_name,
        fa.source_orcid,
        fa.alt_name,
        b.id AS matched_author_id,
        b.display_name AS matched_display_name,
        b.orcid AS matched_orcid
    FROM foreign_alts fa
    INNER JOIN openalex.authors.openalex_authors b
        ON LOWER(TRIM(fa.alt_name)) = LOWER(TRIM(b.display_name))
        AND fa.source_author_id != b.id
    WHERE b.orcid IS NOT NULL
    LIMIT 50
""", size="xlarge")
print(f"Cross-author name matches (foreign alt matches another ORCID author's display_name): {len(cross_match)} rows")
print(cross_match.to_string(index=False))

In [ ]:
# Count total cross-author matches (not just sample)
cross_match_count = run_query_df("""
    WITH foreign_alts AS (
        SELECT
            a.id AS source_author_id,
            alt.col AS alt_name
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW EXPLODE(a.display_name_alternatives) alt
        INNER JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(alt.col) = pn.raw_author_name
        WHERE a.orcid IS NOT NULL
          AND a.parsed_longest_name.last IS NOT NULL
          AND pn.parsed_name.last IS NOT NULL
          AND a.parsed_longest_name.last != pn.parsed_name.last
    )
    SELECT
        COUNT(*) AS total_cross_matches,
        COUNT(DISTINCT fa.source_author_id) AS source_authors_affected
    FROM foreign_alts fa
    INNER JOIN openalex.authors.openalex_authors b
        ON LOWER(TRIM(fa.alt_name)) = LOWER(TRIM(b.display_name))
        AND fa.source_author_id != b.id
    WHERE b.orcid IS NOT NULL
""", size="xlarge")
print("Total cross-author match counts (ORCID subset):")
print(cross_match_count.to_string(index=False))

## Step 8: Combined Summary

Single-pass query computing all flags per author for a unified scope summary.

In [ ]:
summary = run_query_df("""
    WITH
    -- Flag 1: Foreign last names in alternatives
    foreign_ln AS (
        SELECT
            a.id,
            COUNT(DISTINCT CASE
                WHEN pn.parsed_name.last IS NOT NULL
                 AND a.parsed_longest_name.last IS NOT NULL
                 AND pn.parsed_name.last != a.parsed_longest_name.last
                THEN pn.parsed_name.last
            END) AS num_foreign_last_names
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW OUTER EXPLODE(a.display_name_alternatives) alt
        LEFT JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(alt.col) = pn.raw_author_name
        WHERE a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
        GROUP BY a.id
    ),
    -- Flag 2: Block key corrupted (longest_name last != display_name last)
    block_corrupt AS (
        SELECT
            a.id,
            CASE
                WHEN a.parsed_longest_name.last IS NOT NULL
                 AND pn_dn.parsed_name.last IS NOT NULL
                 AND a.parsed_longest_name.last != pn_dn.parsed_name.last
                THEN 1 ELSE 0
            END AS block_key_corrupted
        FROM openalex.authors.openalex_authors a
        LEFT JOIN openalex.authors.parsed_names_lookup pn_dn
            ON TRIM(a.display_name) = pn_dn.raw_author_name
    ),
    -- Flag 3: High Levenshtein distance (any alt >0.6 normalized distance from display_name)
    high_lev AS (
        SELECT
            a.id,
            MAX(
                CASE
                    WHEN GREATEST(LENGTH(a.display_name), LENGTH(alt.col)) > 0
                     AND levenshtein(LOWER(a.display_name), LOWER(alt.col))
                         / GREATEST(LENGTH(a.display_name), LENGTH(alt.col)) > 0.6
                    THEN 1 ELSE 0
                END
            ) AS has_high_lev_alt
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW EXPLODE(a.display_name_alternatives) alt
        WHERE a.display_name_alternatives IS NOT NULL
          AND SIZE(a.display_name_alternatives) > 0
        GROUP BY a.id
    ),
    -- Combine all flags
    combined AS (
        SELECT
            a.id,
            a.orcid IS NOT NULL AS has_orcid,
            COALESCE(fl.num_foreign_last_names, 0) >= 1 AS has_foreign_ln,
            COALESCE(bc.block_key_corrupted, 0) = 1 AS has_block_corruption,
            COALESCE(hl.has_high_lev_alt, 0) = 1 AS has_high_lev
        FROM openalex.authors.openalex_authors a
        LEFT JOIN foreign_ln fl ON a.id = fl.id
        LEFT JOIN block_corrupt bc ON a.id = bc.id
        LEFT JOIN high_lev hl ON a.id = hl.id
    )
    SELECT
        COUNT(*) AS total_authors,
        -- By signal
        COUNT_IF(has_foreign_ln) AS with_foreign_ln,
        COUNT_IF(has_block_corruption) AS with_block_corruption,
        COUNT_IF(has_high_lev) AS with_high_lev,
        -- Any signal
        COUNT_IF(has_foreign_ln OR has_block_corruption OR has_high_lev) AS any_signal,
        -- Overlap
        COUNT_IF(has_foreign_ln AND has_block_corruption) AS foreign_ln_and_block,
        COUNT_IF(has_foreign_ln AND has_high_lev) AS foreign_ln_and_high_lev,
        COUNT_IF(has_block_corruption AND has_high_lev) AS block_and_high_lev,
        COUNT_IF(has_foreign_ln AND has_block_corruption AND has_high_lev) AS all_three,
        -- ORCID breakdown
        COUNT_IF(has_orcid) AS total_orcid,
        COUNT_IF(has_orcid AND has_foreign_ln) AS orcid_foreign_ln,
        COUNT_IF(has_orcid AND has_block_corruption) AS orcid_block_corruption,
        COUNT_IF(has_orcid AND has_high_lev) AS orcid_high_lev,
        COUNT_IF(has_orcid AND (has_foreign_ln OR has_block_corruption OR has_high_lev)) AS orcid_any_signal
    FROM combined
""", size="xlarge")
print("Combined pollution summary:")
for col in summary.columns:
    print(f"  {col}: {summary[col].iloc[0]:,}")

In [ ]:
# Verify known example shows up in signals
verify = run_query_df("""
    WITH
    foreign_ln AS (
        SELECT
            a.id,
            COUNT(DISTINCT CASE
                WHEN pn.parsed_name.last IS NOT NULL
                 AND a.parsed_longest_name.last IS NOT NULL
                 AND pn.parsed_name.last != a.parsed_longest_name.last
                THEN pn.parsed_name.last
            END) AS num_foreign_last_names
        FROM openalex.authors.openalex_authors a
        LATERAL VIEW OUTER EXPLODE(a.display_name_alternatives) alt
        LEFT JOIN openalex.authors.parsed_names_lookup pn
            ON TRIM(alt.col) = pn.raw_author_name
        WHERE a.id = 5059085934
        GROUP BY a.id
    ),
    block_corrupt AS (
        SELECT
            a.id,
            a.parsed_longest_name.last AS longest_last,
            pn_dn.parsed_name.last AS display_last,
            a.parsed_longest_name.last != pn_dn.parsed_name.last AS block_key_corrupted
        FROM openalex.authors.openalex_authors a
        LEFT JOIN openalex.authors.parsed_names_lookup pn_dn
            ON TRIM(a.display_name) = pn_dn.raw_author_name
        WHERE a.id = 5059085934
    )
    SELECT
        fl.id,
        fl.num_foreign_last_names,
        bc.longest_last,
        bc.display_last,
        bc.block_key_corrupted
    FROM foreign_ln fl
    JOIN block_corrupt bc ON fl.id = bc.id
""", size="xlarge")
print("Verification for author 5059085934:")
print(verify.to_string(index=False))